In [1]:
### Take pdf as input -> save it 
### Take this pdf and get the text
### 

# from flask import Flask
# from threading import Thread
# from pypdf import PdfReader
# from flask import Flask, request, render_template_string, jsonify
# from werkzeug.utils import secure_filename
# import os
# app = Flask(__name__)
# # Configuration
# UPLOAD_FOLDER = os.path.expanduser('~')  # Home directory
# MAX_FILE_SIZE = 16 * 1024 * 1024  # 16MB
# ALLOWED_EXTENSIONS = {'pdf'}

# app.config['UPLOAD_FOLDER'] = UPLOAD_FOLDER
# app.config['MAX_CONTENT_LENGTH'] = MAX_FILE_SIZE

# def PDF2text(pdf_file_path):
#     reader = PdfReader(pdf_file_path)
#     text = ""
#     for page in reader.pages:
#         text += page.extract_text() 
#     return text
# @app.route('/')
# def home():
#     my_text = PDF2text("./Maha_Paper.pdf")
#     return my_text

In [ ]:
from flask import Flask
from threading import Thread
from pypdf import PdfReader
from flask import Flask, request, render_template_string, jsonify
from werkzeug.utils import secure_filename
from supermemory import Supermemory
import os
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

app = Flask(__name__)
# Configuration
UPLOAD_FOLDER = os.path.expanduser('~/databackup/Timepass/')  # Home directory
MAX_FILE_SIZE = 16 * 1024 * 1024  # 16MB
ALLOWED_EXTENSIONS = {'pdf'}
app.config['UPLOAD_FOLDER'] = UPLOAD_FOLDER
app.config['MAX_CONTENT_LENGTH'] = MAX_FILE_SIZE

client = Supermemory(
                api_key="sm_3A5a9kAgeJBLn4dMDX8qh2_ZLuNlMSesQMOgstBiWTKYvPqcvkMvwXpOyLfHeUIsqjmjhhqqXcSvjqrbxLqlXIV",
                base_url="https://api.supermemory.ai/"
            )  
def Load_models():
    # Load your 8B model and tokenizer (replace with your model)
    facebook_opt_8_billion = "facebook/opt-8.3b"  # example; replace with actual 8B model
    tokenizer = AutoTokenizer.from_pretrained(facebook_opt_8_billion)
    model = AutoModelForCausalLM.from_pretrained(facebook_opt_8_billion)
    Models = {}
    Models[facebook_opt_8_billion] = (tokenizer,model)
    return Models
    
def PDF2text(pdf_file_path):
    reader = PdfReader(pdf_file_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() 
    return text
def allowed_file(filename):
    return '.' in filename and \
           filename.rsplit('.', 1)[1].lower() in ALLOWED_EXTENSIONS
@app.route('/')
def index():
    # Serve the HTML from a file or render_template_string
    with open('index.html', 'r') as f:
        return render_template_string(f.read())
@app.route('/upload', methods=['POST'])
def upload_file():
    if 'file' not in request.files:
        return jsonify({'error': 'No file provided'}), 400
    file = request.files['file']
    if file.filename == '':
        return jsonify({'error': 'No file selected'}), 400
    if not allowed_file(file.filename):
        return jsonify({'error': 'Only PDF files are allowed'}), 400
    try:
        # Secure the filename and save
        filename = secure_filename(file.filename)
        filepath = os.path.join(app.config['UPLOAD_FOLDER'], filename)
        file.save(filepath)
        extracted_text = PDF2text(filepath)
        # give the data input to supermemory
        response = client.memories.add(content = extracted_text,
                                       container_tag=filename,
                                       )
        return jsonify({
            'success': True,
            'message': f'File uploaded successfully to {filepath}',
            'filename': filename,
            'path': filepath
        }), 200
    except Exception as e:
        return jsonify({'error': f'Upload failed: {str(e)}'}), 500
    
@app.route('/query', methods=['POST'])
def query_pdf():
     
    text_input = request.data.decode('utf-8')

    print(f"User sent text: {text_input}")

    if not text_input:
        return jsonify({'error': 'No text provided'}), 400 
    
    response =  client.search.documents(q=text_input)
    
    prompt = (
        "Here are some relevant facts:\n" +
        "\n".join(f"- {snippet}" for snippet in response) +
        "\n\nBased on the above, answer the following question:\n" +
        text_input
    )
    Models = Load_models()
    Responses = {}
    for Model in Models:
        tokenizer, model = Model
        inputs = tokenizer(prompt, return_tensors="pt")
        # Generate output
        outputs = model.generate(**inputs, max_new_tokens=100)
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        Responses[model] = response
        print(response)
     
    return jsonify({
        'success': True,
        'Responses': Responses
    }), 200
def run_app():
    app.run(host='0.0.0.0', port=5000)
# Start Flask app in a new thread
Thread(target=run_app).start()

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://10.218.100.158:5000
Press CTRL+C to quit
10.218.100.28 - - [01/Nov/2025 21:02:16] "GET / HTTP/1.1" 200 -
